# PI4 — Ingestão e QA do Censo Escolar 2023–2025

Este notebook é a **execução humana reproduzível** do primeiro pipeline de dados do PI4.

## Antes de executar

1. Abra este notebook no Google Colab.
2. Use **Arquivo → Salvar uma cópia no Drive** para manter uma cópia executada com os outputs.
3. Confirme que a pasta compartilhada do projeto está acessível em `Meu Drive` (um atalho para a pasta compartilhada é suficiente).
4. Execute as células em ordem, de cima para baixo, sem editar os arquivos da pasta `1_fonte_original`.
5. Ao final, mantenha os outputs das células salvos: eles serão parte da evidência de reprodutibilidade do projeto.


## 1. Ambiente e Google Drive

Esta etapa monta o Drive, identifica a pasta do projeto e registra informações básicas do ambiente de execução.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import platform
import subprocess
import sys
import time
import pandas as pd
from IPython.display import display

PROJECT_NAME = 'UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá'
candidates = [
    Path('/content/drive/MyDrive') / PROJECT_NAME,
    Path('/content/drive/My Drive') / PROJECT_NAME,
]
PROJECT_ROOT = next((p for p in candidates if p.exists()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Pasta do projeto não encontrada no Meu Drive. '
        'No Google Drive, adicione um atalho da pasta compartilhada do PI4 ao Meu Drive e execute novamente.'
    )

DATA_ROOT = PROJECT_ROOT / '01_Dados'
SOURCE = DATA_ROOT / '1_fonte_original'
OUTPUT_ROOT = DATA_ROOT / '2_tratamentos_dados' / 'base_analitica'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

environment = pd.DataFrame([{
    'executado_em': datetime.now().isoformat(timespec='seconds'),
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'projeto': str(PROJECT_ROOT),
}])
display(environment)


## 2. Código oficial do projeto

O Colab usa diretamente a versão publicada na `main` do repositório oficial. O commit usado na execução fica registrado abaixo.


In [ ]:
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0, '/content/pi4_repo')

REPO_COMMIT = subprocess.check_output(
    ['git', '-C', '/content/pi4_repo', 'rev-parse', 'HEAD'], text=True
).strip()
print('Commit executado:', REPO_COMMIT)

from src.censo_pipeline import (
    build_panel,
    validate_panel,
    qa_summary,
    variable_qa,
    save_outputs,
)


## 3. Conferência das fontes

A célula abaixo confirma que os quatro arquivos necessários existem no Drive e registra seus tamanhos. Nenhum arquivo bruto é alterado.


In [ ]:
sources = {
    '2023_zip': SOURCE / '2023' / 'microdados_censo_escolar_2023.zip',
    '2024_zip': SOURCE / '2024' / 'microdados_censo_escolar_2024.zip',
    '2025_escola': SOURCE / '2025' / 'Tabela_Escola_2025_V2.csv',
    '2025_matricula': SOURCE / '2025' / 'Tabela_Matricula_2025_V2.csv',
}

source_info = pd.DataFrame([
    {
        'fonte': name,
        'arquivo': path.name,
        'existe': path.exists(),
        'tamanho_mb': round(path.stat().st_size / 1024**2, 2) if path.exists() else None,
    }
    for name, path in sources.items()
])
display(source_info)
assert source_info['existe'].all(), 'Há arquivo-fonte ausente. Verifique a estrutura no Drive.'


## 4. Construção do painel escola-ano

Aqui ocorre a leitura dos microdados, o filtro das escolas ativas de São Paulo, a harmonização 2023–2025 e a junção Escola × Matrícula de 2025.


In [ ]:
started = time.time()
panel = build_panel(
    sources['2023_zip'],
    sources['2024_zip'],
    school_2025=sources['2025_escola'],
    matricula_2025=sources['2025_matricula'],
)
validate_panel(panel)
elapsed_seconds = round(time.time() - started, 1)

resumo = qa_summary(panel)
display(resumo)
print(f'Tempo de execução do pipeline: {elapsed_seconds}s')

# Valores já validados no checkpoint técnico anterior.
expected = {
    '2023': {'escolas': 30580, 'municipios': 645},
    '2024': {'escolas': 30746, 'municipios': 645},
    '2025': {'escolas': 30817, 'municipios': 645},
}
indexed = resumo.assign(NU_ANO_CENSO=resumo['NU_ANO_CENSO'].astype(str)).set_index('NU_ANO_CENSO')
for ano, exp in expected.items():
    assert int(indexed.loc[ano, 'escolas']) == exp['escolas'], f'Divergência em escolas de {ano}'
    assert int(indexed.loc[ano, 'municipios']) == exp['municipios'], f'Divergência em municípios de {ano}'
print('CHECKPOINT: números estruturais reproduzidos com sucesso.')


## 5. QA das variáveis

Mantemos nulos como nulos quando a fonte não informa o valor; não os transformamos automaticamente em zero.


In [ ]:
qa_vars = variable_qa(panel)
qa_nulos = qa_vars.pivot(index='variavel', columns='ano', values='pct_nulo').reset_index()
display(qa_nulos)


## 6. Conferência de Guaratinguetá

Este quadro é um primeiro resultado descritivo do município-foco. A EDA completa será feita em etapa posterior.


In [ ]:
guara = panel[panel['CO_MUNICIPIO'] == '3518404'].copy()
guara_summary = (
    guara.groupby('NU_ANO_CENSO', dropna=False)
    .agg(
        escolas=('CO_ENTIDADE', 'nunique'),
        escolas_sem_matricula=('QT_MAT_BAS', lambda s: int(s.isna().sum())),
        matriculas=('QT_MAT_BAS', lambda s: int(s.sum(min_count=1)) if s.notna().any() else None),
    )
    .reset_index()
)
display(guara_summary)


## 7. Materialização da execução no Drive

Cada execução cria uma subpasta própria dentro de `base_analitica`. Isso preserva a evidência da rodada e evita sobrescrever silenciosamente uma execução anterior.


In [ ]:
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = OUTPUT_ROOT / f'execucao_colab_{RUN_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

paths = save_outputs(panel, RUN_DIR)
source_info.to_csv(RUN_DIR / 'fontes_execucao.csv', index=False, encoding='utf-8')
guara_summary.to_csv(RUN_DIR / 'resumo_guaratingueta.csv', index=False, encoding='utf-8')

manifest = pd.DataFrame([{
    'execucao_id': RUN_ID,
    'executado_em': datetime.now().isoformat(timespec='seconds'),
    'python': platform.python_version(),
    'pandas': pd.__version__,
    'repo_commit': REPO_COMMIT,
    'tempo_pipeline_s': elapsed_seconds,
    'linhas_painel': len(panel),
    'linhas_guaratingueta': len(guara),
}])
manifest.to_csv(RUN_DIR / 'manifesto_execucao.csv', index=False, encoding='utf-8')

generated = pd.DataFrame([
    {'arquivo': p.name, 'tamanho_kb': round(p.stat().st_size / 1024, 1)}
    for p in sorted(RUN_DIR.iterdir()) if p.is_file()
])
display(manifest)
display(generated)
print('Arquivos gravados em:', RUN_DIR)


## Evidências a registrar antes de encerrar

Não limpe os outputs do notebook. Para o checkpoint do PI, registre:

1. **Print 1 — fontes:** resultado da seção 3 mostrando os quatro arquivos encontrados.
2. **Print 2 — QA estrutural:** tabela da seção 4 + mensagem `CHECKPOINT: números estruturais reproduzidos com sucesso.`
3. **Print 3 — Guaratinguetá:** tabela da seção 6.
4. **Print 4 — arquivos gerados:** `manifesto_execucao` e lista de arquivos da seção 7.
5. Salve esta cópia do notebook no Drive **com os outputs preservados**.

Quando terminar, não mova nem renomeie os arquivos gerados. A validação P04 usará exatamente essa pasta de execução.
